In [1]:
%load_ext autoreload
%autoreload 2

# Trade Strategy 20250419

## Description
Combine various individual strategies and assets to create a portfolio.

## Libraries

In [6]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm
from utils.data_fetcher import TickerPriceDataFetcher
import os
from indicators.volatility import calculate_atr, calculate_forward_vol
from indicators.returns import calculate_returns, calculate_forward_returns
from utils.helpers import PercentileScaler
import datetime

## Data

In [ ]:
class Asset:
    def __init__(
        self, 
        ticker,
        vol_ticker=None,
        lookforward_period=21,
        lookback_periods=[21, 100, 200, 252],
        rebalance_period=21,
        atr_period=21,
    ):
        # Tickers and Parameters
        self.ticker = ticker
        self.vol_ticker = vol_ticker
        self.lookforward_period = lookforward_period
        self.lookback_periods = lookback_periods
        self.rebalance_period = rebalance_period
        self.atr_period = atr_period

    
    def fetch_price_data(self, ticker):
        """
        Fetch price data from the price_data folder as DataFrame.
        """
        price_data_folder = './data/raw/price/'
        try:
            price_data_file = [f for f in os.listdir(price_data_folder) if f.split('_')[0] == ticker][0]
        except:
            print(f'No price data for {ticker}')
            return None
        return pd.read_pickle(os.path.join(price_data_folder, price_data_file)).copy()
    

    def process_data(self):
        # DataFrames
        self.raw_price_data = self.fetch_price_data(self.ticker)
        self.data = self.fetch_price_data(self.ticker)
        self.vol_data = self.fetch_price_data(self.vol_ticker)

        # Live Trading
        self.current_date = self.raw_price_data.index.to_list()[-1]
        self.current_price = self.raw_price_data['close'].to_list()[-1]

        # Calculate indicators
        self.data['returns'] = self.data['close'].pct_change()
        self.data['vol_forward'] = calculate_forward_vol(self.data, self.lookforward_period)
        self.data['ret_forward'] = calculate_forward_returns(self.data, self.lookforward_period)
        self.data['ATR'] = calculate_atr(self.data, self.atr_period)
        self.data['var_forward'] = self.data['vol_forward'] ** 2


## Load Data

In [8]:
current_date = datetime.datetime.now().date()
current_date = current_date.strftime("%Y%m%d")
lookforward_period = 21
lookback_periods=[21, 100, 200, 252]
rebalance_period=21
atr_period=21

asset_configs = [
    {
        'ticker': 'SPY',
        'vol_ticker': '^VIX',
    },
    {
        'ticker': 'GLD',
        'vol_ticker': '^GVX',
    },
    {
        'ticker': 'TLT',
        'vol_ticker': '^VXTLT',
    },
]

tickers_to_fetch = ['SHV']
assets = dict()
for config in asset_configs:
    assets[config['ticker']] = Asset(
        ticker=config['ticker'],
        vol_ticker=config['vol_ticker'],
        lookforward_period=lookforward_period,
        lookback_periods=lookback_periods,
        rebalance_period=rebalance_period,
        atr_period=atr_period,
    )

    tickers_to_fetch.append(config['ticker'])
    tickers_to_fetch.append(config['vol_ticker'])

fetcher = TickerPriceDataFetcher()
price_data = fetcher.load_data(ticker_list=tickers_to_fetch)
cash_asset = {'ticker': 'SHV', 'data': price_data['SHV']}



    

    

2025-04-20 18:59:57,561 - utils.data_fetcher - INFO - Loading data for ['SHV', 'SPY', '^VIX', 'GLD', '^GVX', 'TLT', '^VXTLT']
2025-04-20 18:59:57,563 - utils.data_fetcher - INFO - Data for SHV already exists
2025-04-20 18:59:57,579 - utils.data_fetcher - INFO - Data for SPY already exists
2025-04-20 18:59:57,592 - utils.data_fetcher - INFO - Data for ^VIX already exists
2025-04-20 18:59:57,603 - utils.data_fetcher - INFO - Data for GLD already exists
2025-04-20 18:59:57,617 - utils.data_fetcher - INFO - Fetching data for ^GVX with yfinance
2025-04-20 18:59:57,660 - yfinance - ERROR - ^GVX: Period 'max' is invalid, must be of the format 1d, 5d, etc.
2025-04-20 18:59:57,693 - utils.data_fetcher - INFO - Fetching data for ^GVX with yahoo_fin
2025-04-20 18:59:57,719 - utils.data_fetcher - ERROR - Failed to fetch data for ^GVX with yahoo_fin
2025-04-20 18:59:57,720 - utils.data_fetcher - ERROR - Failed to fetch data for ^GVX
2025-04-20 18:59:57,720 - utils.data_fetcher - ERROR - Failed to f